<a href="https://colab.research.google.com/github/bahmedx/733/blob/main/Hands_On_Project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Hands-On Project 2**
### **Data Visualization with Python**
This project analyzes the Seaborn Diamonds dataset using Python visualization libraries to identify trends, patterns, and relationships influencing diamond prices.

## **Import Libraries**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization settings
sns.set_style("whitegrid")
sns.set_palette("pastel")

print("Libraries loaded successfully.")

## **Load and Inspect the Dataset**

In [ ]:
# Load Seaborn dataset
df = sns.load_dataset('diamonds')

# Dataset overview
print("Dataset Shape:")
print(df.shape)

print("\nFirst 5 Rows:")
display(df.head())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

## **Sample Records**

In [ ]:
df.sample(5)

# **Essential Plots**

### **Average Diamond Price by Cut**

In [ ]:
plt.figure(figsize=(8,5))

avg_price = (
    df.groupby('cut', observed=False)['price']
      .mean()
      .sort_values()
)

avg_price.plot(kind='bar', color='skyblue')

plt.title("Average Diamond Price by Cut")
plt.xlabel("Cut Quality")
plt.ylabel("Average Price ($)")
plt.xticks(rotation=45)

plt.show()

### **Distribution of Diamond Prices**

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(df['price'], bins=30, kde=True)

plt.title("Distribution of Diamond Prices")
plt.xlabel("Price ($)")
plt.ylabel("Frequency")

plt.show()

### **Carat Weight vs Average Price and Carat Weight vs Average Diamond Length by Cut**

In [ ]:
# Average price by rounded carat weight and cut
price_summary = (
    df.assign(carat_round=df['carat'].round(1))
      .groupby(['carat_round', 'cut'], observed=False)['price']
      .mean()
      .reset_index()
)

length_summary = (
    df.assign(carat_round=df['carat'].round(1))
      .groupby(['carat_round', 'cut'], observed=False)['x']
      .mean()
      .reset_index()
)

pastel_palette = sns.color_palette("pastel", n_colors=5)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Carat Weight vs Price

sns.scatterplot(
    data=price_summary,
    x='carat_round',
    y='price',
    hue='cut',
    palette=pastel_palette,
    s=60,
    ax=ax[0]
)

ax[0].set_title('Carat Weight vs Average Price')
ax[0].set_xlabel('Carat Weight')
ax[0].set_ylabel('Average Price ($)')

# Carat Weight vs Diamond Length

sns.scatterplot(
    data=length_summary,
    x='carat_round',
    y='x',
    hue='cut',
    palette=pastel_palette,
    s=60,
    ax=ax[1]
)

ax[1].set_title('Carat Weight vs Average Diamond Length')
ax[1].set_xlabel('Carat Weight')
ax[1].set_ylabel('Diamond Length (mm)')

# Shared legend
for a in ax:
    if a.get_legend() is not None:
        a.get_legend().remove()

handles, labels = ax[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.02),
    ncol=5,
    frameon=False,
    title=None
)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

## **Median Price by Cut and Carat Range**

In [ ]:
df['carat_range'] = pd.cut(
    df['carat'],
    bins=[0.20, 1, 2, 3, 4, 5.1],
    labels=[
        '0.20-1.00',
        '1.01-2.00',
        '2.01-3.00',
        '3.01-4.00',
        '4.01-5.00'
    ]
)

# Display frequency table
print("Diamond Count by Carat Range and Cut")
display(pd.crosstab(df['carat_range'], df['cut']))

# Median price by cut and carat range
summary = (
    df.groupby(['cut', 'carat_range'], observed=False)['price']
      .median()
      .reset_index()
)

plt.figure(figsize=(14,6))

sns.barplot(
    data=summary,
    x='cut',
    y='price',
    hue='carat_range',
    palette='viridis'
)

plt.title('Median Price by Cut and Carat Range')
plt.xlabel('Cut')
plt.ylabel('Median Price ($)')

plt.legend(
    title='Carat Range',
    loc='lower center',
    bbox_to_anchor=(0.5, -0.25),
    ncol=5,
    frameon=False
)

plt.tight_layout()
plt.show()

# **Statistical Visualizations**

### **Price Distribution by Cut Category (Violin Plot)**

In [ ]:
plt.figure(figsize=(10,6))

sns.violinplot(
    data=df,
    x='cut',
    y='price'
)

plt.title("Price Distribution by Cut")
plt.xlabel("Cut")
plt.ylabel("Price ($)")

plt.show()

### **Correlation Matrix of Carat Weight, Depth, and Price**

In [ ]:
corr_vars = df[['carat', 'depth', 'price']]

plt.figure(figsize=(6,4))

sns.heatmap(
    corr_vars.corr(),
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title('Correlation Matrix')
plt.show()

### **Price by Clarity and Price by Color**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# Price by Clarity
sns.boxplot(ax=axes[0], x='clarity', y='price', data=df, hue='clarity', palette='Blues', legend=False)
axes[0].set_title('Price by Clarity')
axes[0].set_xlabel('Clarity Grade')
axes[0].set_ylabel('Price ($)')

# Price by Color
sns.boxplot(ax=axes[1], x='color', y='price', data=df, hue='color', palette='Greens', legend=False)
axes[1].set_title('Price by Color')
axes[1].set_xlabel('Color Grade')

plt.tight_layout()
plt.show()

### **Relationships Among Carat Weight, Price, and Diamond Length by Cut**

In [ ]:
pairplot_df = (
    df[['carat', 'price', 'x', 'cut']]
      .rename(columns={
          'carat': 'Carat Weight',
          'price': 'Price ($)',
          'x': 'Diamond Length (mm)'
      })
      .sample(1000, random_state=42)
)

pastel_colors = sns.color_palette(
    "pastel",
    n_colors=len(pairplot_df['cut'].cat.categories)
)

# Create pairplot
g = sns.pairplot(
    pairplot_df,
    hue='cut',
    palette=pastel_colors,
    corner=True,
    height=3.5,
    plot_kws={
        'alpha': 0.6,
        's': 25
    },
    diag_kws={
        'fill': True
    }
)

# Remove Seaborn's default legend on the right
if g._legend is not None:
    g._legend.remove()

# Compact layout
g.figure.subplots_adjust(
    top=0.92,
    bottom=0.15,
    hspace=0.12,
    wspace=0.12
)

# Create custom single-row legend at the bottom
handles = [
    plt.Line2D(
        [0], [0],
        marker='o',
        linestyle='',
        markersize=8,
        markerfacecolor=color,
        markeredgecolor=color,
        label=label
    )
    for label, color in zip(
        pairplot_df['cut'].cat.categories,
        pastel_colors
    )
]

g.figure.legend(
    handles=handles,
    labels=[str(c) for c in pairplot_df['cut'].cat.categories],
    loc='lower center',
    bbox_to_anchor=(0.5, 0.02),
    ncol=5,
    frameon=False
)

# Main title
g.figure.suptitle(
    'Relationships Among Carat Weight, Price, and Diamond Length by Cut',
    fontsize=14,
    fontweight='bold',
    y=0.98
)

plt.show()